In [ ]:
# Smoke test for JengaAllegro-v1 (single floating Allegro hand). Run on the dev node in the ManiSkill .venv.
# Written but not yet run: expect to fix small things (init pose, gains) on first contact with the sim.
import numpy as np, torch
from api import maniskill
from IPython.display import display
from PIL import Image

env = maniskill.make_env("JengaAllegro-v1", num_envs=1, render=True, obs_mode="state", sim_backend="physx_cpu")
obs, info = env.reset(seed=0)
agent = env.unwrapped.agent
print("action space:", env.action_space.shape, "(expect 22 = 6 root + 16 fingers)")
print("control mode:", env.unwrapped.control_mode)
print("qpos dof:", agent.robot.max_dof, "joints:", [j.name for j in agent.robot.get_active_joints()])
print("palm pose:", agent.palm_link.pose.p, "tips:", agent.tip_poses.shape)

In [ ]:
# Does the hand hold still? With zero action (delta controller) it should hover, not sag or drift.
p0 = agent.palm_link.pose.p.clone()
for _ in range(100):
    obs, r, term, trunc, info = env.step(torch.zeros(env.action_space.shape))
print("palm drift after 100 zero-action steps (m):", (agent.palm_link.pose.p - p0).norm().item())
display(Image.fromarray(env.render().squeeze(0).cpu().numpy()))

In [ ]:
# Do the controllers move the right joints? Command root_pos +x (first 3 dims), then close the fingers (last 16 dims).
def run(act, n=40):
    a = torch.zeros(env.action_space.shape); a[:] = torch.as_tensor(act)
    for _ in range(n):
        env.step(a)
p0 = agent.palm_link.pose.p.clone()
act = np.zeros(22); act[0] = 1.0
run(act); print("moved +x by", (agent.palm_link.pose.p - p0)[0, 0].item(), "m (root_pos max delta/step = 0.02)")
act = np.zeros(22); act[6:] = 1.0
run(act, 20)
print("finger qpos after closing:", agent.robot.get_qpos()[0, 6:].cpu().numpy().round(2))
display(Image.fromarray(env.render().squeeze(0).cpu().numpy()))

In [ ]:
# Does the score pipeline still work? Teleport the blocks into the design: score should equal max_score.
env.unwrapped.place_design()
for _ in range(60):
    obs, r, term, trunc, info = env.step(torch.zeros(env.action_space.shape))
print("score", info["score"].item(), "/", info["max_score"].item())
env.close()

In [ ]:
# Cameras: one fixed workspace camera (base_camera, from the env) + one palm camera (palm_camera, on the hand).
# obs_mode="rgb" renders sensors; "state" (used above) does not. If palm_camera looks wrong (inside the hand,
# wrong direction), tune FloatingAllegroHandRight.palm_camera_pose (palm-frame pose) and re-run this cell.
cam_env = maniskill.make_env("JengaAllegro-v1", num_envs=1, render=True, obs_mode="rgbd", sim_backend="physx_cpu")
obs, _ = cam_env.reset(seed=0)
for _ in range(20):
    obs, *_ = cam_env.step(torch.zeros(cam_env.action_space.shape))
print({k: {c: tuple(v.shape) for c, v in d.items()} for k, d in obs["sensor_data"].items()})
frames = [obs["sensor_data"][k]["rgb"][0].cpu().numpy() for k in ("base_camera", "palm_camera")]
display(Image.fromarray(np.concatenate(frames, axis=1)))
cam_env.close()